# Results Analysis

This notebook loads experiment outputs, builds summary tables, runs statistical analysis, and generates Phase 7 figures.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from experiments.generate_figures import load_epoch_curves, load_results  # noqa: E402
from experiments.statistical_analysis import build_full_significance_table  # noqa: E402

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

In [2]:
results_df = load_results(RESULTS_DIR)
epoch_df = load_epoch_curves(RESULTS_DIR)
results_df.head()

,path,dataset,model,heuristic,condition,seed,phase_history,auc,ap,mrr,hits@10,hits@50,hits@100
0,/home/oswin/CompSci/IAI_Research_Project/gnn_c...,cora,gcn,cn,abl-5,3,"[{'epoch': 0, 'phase_name': 'easy_only'}, {'ep...",0.800964,0.809062,0.099274,0.018975,0.091082,0.182163
1,/home/oswin/CompSci/IAI_Research_Project/gnn_c...,cora,gcn,cn,abl-5,3,"[{'epoch': 0, 'phase_name': 'easy_only'}, {'ep...",0.800964,0.809062,0.099274,0.018975,0.091082,0.182163
2,/home/oswin/CompSci/IAI_Research_Project/gnn_c...,cora,gcn,cn,abl-5,4,"[{'epoch': 0, 'phase_name': 'easy_only'}, {'ep...",0.868735,0.865753,0.160265,0.018975,0.094877,0.182163
3,/home/oswin/CompSci/IAI_Research_Project/gnn_c...,cora,gcn,None,curriculum,0,[],0.916969,0.925076,0.322204,0.018975,0.094877,0.189753
4,/home/oswin/CompSci/IAI_Research_Project/gnn_c...,cora,gcn,cn,curriculum,0,"[{'epoch': 0, 'phase_name': 'easy_only'}, {'ep...",0.875994,0.875450,0.159059,0.018975,0.092979,0.184061


In [3]:
summary_columns = [
    col
    for col in ["auc", "ap", "mrr", "heart_mrr", "heart_hits@10", "heart_hits@50"]
    if col in results_df.columns
]
summary_table = (
    results_df.groupby(["dataset", "model", "condition"], dropna=False)[
        summary_columns
    ].agg(["mean", "std"])
    if not results_df.empty
    else pd.DataFrame()
)
summary_table

auc                  ap                 mrr  \
                              mean       std      mean       std      mean   
dataset model condition                                                      
cora    gcn   abl-5       0.823555  0.039128  0.827959  0.032730  0.119604   
              curriculum  0.896482  0.028974  0.900263  0.035091  0.240631   

                                    
                               std  
dataset model condition             
cora    gcn   abl-5       0.035213  
              curriculum  0.115361

In [4]:
significance_table = build_full_significance_table(results_df)
significance_table

""


In [5]:
from experiments.generate_figures import (
    plot_ablation_study,
    plot_competence_progression,
    plot_heart_gap,
    plot_heuristic_comparison,
    plot_learning_curves,
    plot_performance_comparison,
    plot_phase_transitions,
)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plot_learning_curves(epoch_df, results_df, FIGURES_DIR)
plot_performance_comparison(results_df, FIGURES_DIR)
plot_heart_gap(results_df, FIGURES_DIR)
plot_heuristic_comparison(results_df, FIGURES_DIR)
plot_phase_transitions(results_df, FIGURES_DIR)
plot_competence_progression(epoch_df, FIGURES_DIR)
plot_ablation_study(results_df, FIGURES_DIR)
sorted(p.name for p in FIGURES_DIR.glob("*.png"))[:10]

['ablation_study.png',
 'competence_progression.png',
 'heart_gap_analysis.png',
 'heuristic_comparison.png',
 'learning_curves_abl-5_cora.png',
 'learning_curves_cora_gcn.png',
 'performance_comparison_cora.png',
 'phase_transitions.png']

<Figure size 800x400 with 0 Axes>

## Interpretation

- The current repository contains smoke-test scale results, not the full multi-seed experiment matrix.
- Summary statistics and significance tests should be treated as preliminary until all Phase 6 runs are completed.
- The notebook still serves as the final analysis pipeline and should scale to the full result set once those runs are available.